# Duyệt ảnh CSV v2.2 trên GPU Kaggle
Bật **GPU + Internet**, add dataset chứa `tool_kaggle_bundle.zip` (hoặc nội dung đã bung). Không cần API key.
Chạy thử `MODE="pilot"` trước; xem chất lượng rồi đổi `MODE="full"`. Mọi output ở `/kaggle/working/tool/`.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, zipfile, json
DATASET_TOOL = None  # VD: Path("/kaggle/input/my-dataset/tool") nếu auto-detect có nhiều kết quả
PREVIOUS_CHECKPOINT = None  # Phiên mới: path tới resume_bundle.zip hoặc thư mục checkpoint đã bung
MODE = "first"  # "first": N dòng đầu; "pilot": 50 dòng cũ; "full": mọi dòng trống
FIRST_ROWS = 2000
RESUME = False  # True để tiếp tục cùng run
RUN_NAME = "qwen_fast2_first2000_b4"  # Run mới; dùng đúng tên này + RESUME=True để tiếp tục
WORKERS = 8
BATCH_SIZE = 4
BATCH_TOKEN_BUDGET = 12288  # Muốn cấu hình cũ: workers=4, batch=2, budget=8192
MAX_HOURS = 8
WORK = Path("/kaggle/working/tool")
WORK.mkdir(parents=True, exist_ok=True)
if DATASET_TOOL is None:
    candidates = [p.parent.parent for p in Path("/kaggle/input").rglob("run_gpu.py")
                  if p.parent.name == "kaggle" and (p.parent.parent/"approve_tool.py").exists()]
    if len(candidates) == 1:
        DATASET_TOOL = candidates[0]
    elif not candidates:
        bundles = list(Path("/kaggle/input").rglob("tool_kaggle_bundle.zip"))
        if len(bundles) != 1:
            raise RuntimeError("Add dataset chứa bundle, hoặc đặt DATASET_TOOL chính xác")
        unpack = Path("/kaggle/working/uploaded_bundle")
        with zipfile.ZipFile(bundles[0]) as z:
            for member in z.infolist():
                target = (unpack/member.filename).resolve()
                if not target.is_relative_to(unpack.resolve()): raise RuntimeError("Unsafe zip path")
            z.extractall(unpack)
        DATASET_TOOL = unpack/"tool"
    else:
        raise RuntimeError("Có nhiều bộ code; điền DATASET_TOOL")
DATASET_TOOL = Path(DATASET_TOOL)
for name in ["approve_tool.py", "matching.py", "source_page.py", "name_overrides.json", "pilot_v1_selection.json",
             "kaggle/run_gpu.py", "kaggle/gpu_backend.py", "kaggle/qwen_policy.py", "kaggle/requirements.txt", "kaggle/restore.py"]:
    dest = WORK/name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATASET_TOOL/name, dest)
INPUT = DATASET_TOOL/"input/general_10m_part_02_copy.csv"
assert INPUT.is_file(), f"Thiếu CSV: {INPUT}"
os.environ["HF_HOME"] = str(WORK/"cache/hf")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Dataset:", DATASET_TOOL)
print("Input:", INPUT)


In [ ]:
# Cài vào môi trường Kaggle; giữ torch/torchvision CUDA có sẵn.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(WORK/"kaggle/requirements.txt")], check=True)
subprocess.run(["nvidia-smi"], check=True)


## Khôi phục checkpoint (chỉ phiên mới)
Để `PREVIOUS_CHECKPOINT=None` khi chạy mới hoặc resume trong cùng working directory.
Không ghi đè output mới hơn bằng bundle cũ.


In [ ]:
if PREVIOUS_CHECKPOINT:
    sys.path.insert(0, str(WORK/"kaggle"))
    from restore import restore
    restore(PREVIOUS_CHECKPOINT, WORK)
    RESUME = True
    print("Restored checkpoint; RESUME=True")


In [ ]:
assert MODE in ("pilot", "first", "full")
OUTPUT = WORK/"outputs"/RUN_NAME
cmd = [sys.executable, str(WORK/"kaggle/run_gpu.py"), "--input", str(INPUT),
       "--work-dir", str(WORK), "--run-name", RUN_NAME, "--max-hours", str(MAX_HOURS)]
if MODE == "pilot":
    cmd += ["--limit", "50", "--selection-file", str(WORK/"pilot_v1_selection.json")]
elif MODE == "first":
    cmd += ["--first", str(FIRST_ROWS)]
else:
    cmd += ["--all"]
# Nhiều record dùng CHUNG một model; batch tự giảm nếu thiếu VRAM.
cmd += ["--workers", str(WORKERS), "--batch-size", str(BATCH_SIZE),
        "--max-pixels", "602112", "--max-input-tokens", "4096",
        "--max-new-tokens", "512", "--batch-token-budget", str(BATCH_TOKEN_BUDGET), "--url-first"]
if (OUTPUT/"manifest.json").exists() and not RESUME:
    raise RuntimeError("Run đã tồn tại: đặt RESUME=True để tiếp tục, hoặc đổi RUN_NAME để chạy mới.")
if RESUME:
    cmd += ["--resume"]
else:
    cmd += ["--overrides", str(WORK/"name_overrides.json")]
print(" ".join(cmd))
# Subprocess gọi model local GPU; không dùng OpenAI API.
proc = subprocess.Popen(cmd)
try:
    returncode = proc.wait()
except KeyboardInterrupt:
    import signal
    proc.send_signal(signal.SIGINT)
    print("Đợi script hoàn tất các record đang chạy và lưu checkpoint...")
    returncode = proc.wait()
print("Exit code:", returncode)
if returncode:
    print("Kiểm tra log lỗi trước khi chạy lại; không tự chuyển lỗi thành nhãn no.")


In [ ]:
summary_path = OUTPUT/"summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))
else:
    print("Chưa có summary; xem lỗi tải model/GPU ở cell trên.")
# Xem 10 quyết định đầu, không load toàn bộ journal vào RAM.
if (OUTPUT/"results.jsonl").exists():
    import itertools
    with (OUTPUT/"results.jsonl").open() as f:
        for line in itertools.islice(f, 10):
            r = json.loads(line)
            print(r["row_index"], r["entity_name"], r["approve"] or "trống", r["reason"])


## Lưu kết quả trước khi hết phiên
Tải `resume_bundle.zip` để tiếp tục phiên khác. CSV có thể chưa hoàn tất: kiểm tra `remaining` trong summary.
Save Version của Kaggle có thể chạy lại notebook; khi cấu hình một lần chạy mới hãy chú ý MODE/RESUME và checkpoint đã attach.


In [ ]:
from IPython.display import FileLink, display
for filename in ("reviewed.csv", "summary.json", "manifest.json", "results.jsonl", "resume_bundle.zip"):
    path = OUTPUT/filename
    if path.exists():
        display(FileLink(str(path)))
